In [1]:
import sys
sys.path.insert(0, "../")
from cancerfoundation.model.module import TransformerModule
import scanpy as sc
import os
import json
from cancerfoundation.loss import LossType
import torch
with open("./save/condtech/args.json", "r") as f:
    model_configs = json.load(f)
with open("./save/condtech/vocab.json", "r") as f:
    vocab = json.load(f)

# model = TransformerModule(
#     ntoken=len(vocab),
#     d_model=model_configs["embsize"],
#     out_dim=
#     nhead=model_configs["nheads"],
#     d_hid=model_configs["d_hid"],
#     nlayers=model_configs["nlayers"],
#     vocab=vocab,
#     dropout=model_configs["dropout"],
#     pad_token=pad_token,
# )
# model = CancerFoundation.load_from_checkpoint(vocab='./save/condtech/vocab.json', checkpoint_path='./save/condtech/epoch_epoch=04.ckpt')

# for model_name in ["bf16", "bigger_vocab", "coarse", "default", "medium", "github", "brain"]:
# for model_name in ["condtech"]:
#     data_dir = "./data/Kim_lung/"
#     dataset = data_dir.split('/')[-2].lower()

#     model_dir = "../model/assets/" + model_name
#     adata_path = data_dir + dataset + ".h5ad"
#     adata = sc.read_h5ad(adata_path)


#     embed_adata = embed(
#         adata_or_file=adata,
#         model_dir=model_dir,
#         batch_key="sample",
#         batch_size=64,
#     )
#     os.makedirs("./data", exist_ok=True)
#     embed_adata.write_h5ad(data_dir + "CancerGPT_" + model_name + "_" + dataset + ".h5ad")

In [2]:
print(model_configs)

{'n_bins': 51, 'max_seq_len': 1200, 'trunc_by_sample': True, 'nlayers': 6, 'nheads': 8, 'embsize': 256, 'd_hid': 512, 'dropout': 0.2, 'n_layers_cls': 3}


In [48]:
model = TransformerModule(
    ntoken=len(vocab),
    d_model=model_configs["embsize"],
    out_dim=1,
    criterion=LossType.MSE,
    nhead=model_configs["nheads"],
    d_hid=model_configs["d_hid"],
    nlayers=model_configs["nlayers"],
    dropout=model_configs["dropout"],
    pad_value=51,
    pad_token_id=1,
    conditions={"technology": 15},
    do_mvc=True
)

In [49]:
print(model)

TransformerModule(
  (encoder): GeneEncoder(
    (embedding): Embedding(28725, 256, padding_idx=1)
    (enc_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (flag_encoder): Embedding(2, 256)
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.2, inplace=False)
    (linear1): Linear(in_features=1, out_features=256, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=256, out_features=256, bias=True)
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (criterion_conditions): CrossEntropyLoss()
  (condition_encoders): ModuleDict(
    (technology): ConditionEncoder(
      (embedding): Embedding(15, 256)
      (enc_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features

In [50]:
state_dict = torch.load("./save/condtech/epoch_epoch=04.ckpt")['state_dict']
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('model._orig_mod.'):
        new_key = k[len('model._orig_mod.'):]
    if 'self_attn.self_attn' in new_key:
        new_key = new_key.replace('self_attn.self_attn', 'self_attn')
        
    new_state_dict[new_key] = v


In [51]:
print(new_state_dict['condition_encoders.technology.embedding.weight'].shape)

torch.Size([15, 256])


In [52]:
torch.serialization.add_safe_globals([LossType])
model.load_state_dict(new_state_dict, strict=True)

<All keys matched successfully>

In [53]:
adata = sc.read_h5ad('./neftel_ss2.h5ad')

In [59]:
print(adata.obs['technology'])

cell_name
MGH100-P5-A01    SmartSeq2
MGH100-P5-A04    SmartSeq2
MGH100-P5-A07    SmartSeq2
MGH100-P5-A10    SmartSeq2
MGH100-P5-B08    SmartSeq2
                   ...    
MGH66-P08-H02    SmartSeq2
MGH66-P08-H06    SmartSeq2
MGH66-P08-H07    SmartSeq2
MGH66-P08-H10    SmartSeq2
MGH66-P08-H11    SmartSeq2
Name: technology, Length: 4338, dtype: category
Categories (1, object): ['SmartSeq2']
